#**Retail Store Sales Analysis and Prediction**

##**Introduction**
The Retail Store Sales Prediction project was developed to analyze customer transaction data and predict customer spending behavior using machine learning techniques. Retail businesses generate large amounts of transactional data every day, and analyzing this data can help organizations improve decision-making, sales strategies, inventory planning, and customer understanding.

This project focuses on the complete data science workflow, including data loading, data cleaning, exploratory data analysis (EDA), feature engineering, predictive modeling, model optimization, and visualization. The dataset contains retail transaction information such as product category, item type, quantity purchased, payment method, location, discount status, and total spending amount.

Machine learning models such as Linear Regression and Ridge Regression were used to predict the Total Spent value. Different visualizations and evaluation metrics were also applied to understand customer purchasing patterns and evaluate model performance.

Overall, this project demonstrates how data analytics and machine learning can be applied in the retail industry to support business intelligence and improve sales prediction accuracy.

##**Project Objectives**
The main objectives of this project are:
1.   To load and preprocess retail transaction
data for analysis and machine learning.
2.   To identify and handle missing values, inconsistent data, and duplicate records.
3. To perform Exploratory Data Analysis (EDA) using different visualizations to understand customer purchasing behavior and sales trends.
4. To create new features that improve the predictive performance of machine learning models.
5. To build predictive models that can estimate customer Total Spent values.
6. To evaluate machine learning models using regression metrics such as MAE, MSE, RMSE, and R-squared.
7. To optimize model performance using hyperparameter tuning and dimensionality reduction techniques.
8. To generate business insights that can help improve retail sales strategies and customer understanding.





##**Problem Statement**

Retail businesses generate large amounts of transaction data every day, but it is difficult to accurately predict customer spending behavior and sales patterns. Without proper analysis, businesses may face challenges in inventory management, sales forecasting, and understanding customer purchasing trends.

This project aims to analyze retail transaction data and use machine learning techniques to predict customer Total Spent values. The project also focuses on identifying important sales trends and customer behavior patterns that can support better business decision-making.

##Import Libraries and Dataset Loading





In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# Make plots look clean
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/maybraniswe/Retail-Store-Sales-Analysis-and-Prediction/refs/heads/main/retail_store_sales.csv')

# Check dataset size
df.shape

Observation:

The dataset is successfully loaded into a pandas DataFrame.
The dataset contains 12,575 rows and 11 columns.
Each row represents one retail transaction record.

##Initial Data Inspection

In [ ]:
# Display first 5 rows
df.head()

Observation:

The first few records show transaction details including Transaction ID, Customer ID, Category, Item, Price Per Unit, Quantity, Total Spent, Payment Method, Location, Transaction Date, and Discount Applied.

In [ ]:
# Display statistical summary for numerical columns
df.describe().round(2)

Observation:

Price Per Unit ranges from 5.00 to 41.00.
Quantity ranges from 1 to 10.
Total Spent ranges from 5.00 to 410.00.
This shows that the numerical values are within a reasonable range.

In [ ]:
# Display full data information
df.info()

Observation:

The dataset has both object and numeric data types.
Transaction Date is currently stored as object, so it needs to be converted to datetime.
Discount Applied is also stored as object and should be converted to boolean/category.

In [ ]:
# Check missing values
df.isnull().sum()

Observation:

Missing values are found in Item, Price Per Unit, Quantity, Total Spent, and Discount Applied.
These missing values need to be handled before analysis and modeling.

##Data Cleaning


In [ ]:
# ==========================================================
# DATA CLEANING
# ==========================================================

# Create a copy of original dataset
clean_df = df.copy()

# Check original shape
print("Original dataset shape:", clean_df.shape)

Observation:

A copy is created so the original dataset remains unchanged.
This is a good practice before performing data cleaning.

In [ ]:
# ==========================================================
# REMOVE DUPLICATE ROWS
# ==========================================================

duplicate_count = clean_df.duplicated().sum()
print("Duplicate rows before cleaning:", duplicate_count)

clean_df = clean_df.drop_duplicates()

print("Duplicate rows after cleaning:", clean_df.duplicated().sum())
print("Dataset shape after removing duplicates:", clean_df.shape)

Observation:

Fully duplicate rows are checked and removed.
This helps avoid repeated records affecting the analysis.

In [ ]:
# ==========================================================
# CONVERT TRANSACTION DATE TO DATETIME
# ==========================================================

clean_df["Transaction Date"] = pd.to_datetime(
    clean_df["Transaction Date"],
    errors="coerce"
)

clean_df["Transaction Date"].dtype

Observation:

Transaction Date is converted from object to datetime format.
This allows future analysis by year, month, or day.

In [ ]:
# ==========================================================
# STANDARDIZE CATEGORICAL TEXT COLUMNS
# ==========================================================

categorical_cols = [
    "Transaction ID",
    "Customer ID",
    "Category",
    "Item",
    "Payment Method",
    "Location",
    "Discount Applied"
]

for col in categorical_cols:
    clean_df[col] = clean_df[col].astype(str).str.strip().str.lower()

 # Convert fake string missing values back to real NaN
clean_df = clean_df.replace(
    ["nan", "none", ""],
    np.nan
)
clean_df.head()


Observation:

Extra spaces are removed from categorical columns.
This helps avoid inconsistent values such as 'Cash ' and 'Cash' being treated as different categories.

In [ ]:
# ==========================================================
# HANDLE MISSING VALUES
# ==========================================================

# Remove rows where both Item and Price Per Unit are missing
clean_df = clean_df.dropna(
    subset=["Item", "Price Per Unit"],
    how="all"
)

# Create reference table from available Item and Price Per Unit
item_price_table = clean_df[["Item", "Price Per Unit"]].dropna().drop_duplicates()

# Fill missing Item using Price Per Unit
price_to_item = item_price_table.drop_duplicates("Price Per Unit") \
                                .set_index("Price Per Unit")["Item"]

clean_df["Item"] = clean_df["Item"].fillna(
    clean_df["Price Per Unit"].map(price_to_item)
)

# Fill missing Price Per Unit using Item
item_to_price = item_price_table.drop_duplicates("Item") \
                                .set_index("Item")["Price Per Unit"]

clean_df["Price Per Unit"] = clean_df["Price Per Unit"].fillna(
    clean_df["Item"].map(item_to_price)
)

# Fill Quantity using median
clean_df["Quantity"] = clean_df["Quantity"].fillna(
    clean_df["Quantity"].median()
)

# Recalculate Total Spent
clean_df["Total Spent"] = clean_df["Price Per Unit"] * clean_df["Quantity"]

# Fill Discount Applied
clean_df["Discount Applied"] = (
    clean_df["Discount Applied"]
    .map({"true": True, "false": False})
    .fillna(False)
    .astype(bool)
)

clean_df.isnull().sum()

Observation:

Missing values were handled using different strategies based on the column type.

Item and Price Per Unit were filled using their relationship.

Quantity was filled using the median value.

Total Spent was recalculated to maintain accuracy.

Discount Applied missing values were filled as False.

In [ ]:
# ==========================================================
# FINAL CLEAN DATA CHECK
# ==========================================================

print("Final dataset shape:", clean_df.shape)

print("\nFinal missing values:")
print(clean_df.isnull().sum())

print("\nFinal data types:")
print(clean_df.dtypes)

clean_df.head()

##Exploratory Data Analysis (EDA)

####Univariate Analysis

In [ ]:
# Histogram with KDE and Boxplot for Total Spent

fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 1]})

sns.histplot(clean_df["Total Spent"], kde=True, bins=30, ax=axes[0])
axes[0].set_title("Distribution of Total Spent")
axes[0].set_xlabel("Total Spent")
axes[0].set_ylabel("Frequency")

sns.boxplot(x=clean_df["Total Spent"], ax=axes[1])
axes[1].set_title("Boxplot of Total Spent")
axes[1].set_xlabel("Total Spent")

plt.tight_layout()
plt.show()


Observation:

The histogram shows how customer spending is distributed.
The KDE line helps show the overall spending pattern.
The boxplot shows the spread of Total Spent and helps identify possible outliers.

In [ ]:
# Histogram with KDE and Boxplot for Price Per Unit

fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 1]})

sns.histplot(clean_df["Price Per Unit"], kde=True, bins=20, ax=axes[0])
axes[0].set_title("Distribution of Price Per Unit")
axes[0].set_xlabel("Price Per Unit")
axes[0].set_ylabel("Frequency")

sns.boxplot(x=clean_df["Price Per Unit"], ax=axes[1])
axes[1].set_title("Boxplot of Price Per Unit")
axes[1].set_xlabel("Price Per Unit")

plt.tight_layout()
plt.show()

Observation:

This plot shows the distribution of item prices.
The boxplot helps identify whether some items have much higher unit prices than others.

In [ ]:
# Histogram with KDE and Boxplot for Quantity

fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 1]})

sns.histplot(clean_df["Quantity"], kde=True, bins=10, ax=axes[0])
axes[0].set_title("Distribution of Quantity")
axes[0].set_xlabel("Quantity")
axes[0].set_ylabel("Frequency")

sns.boxplot(x=clean_df["Quantity"], ax=axes[1])
axes[1].set_title("Boxplot of Quantity")
axes[1].set_xlabel("Quantity")

plt.tight_layout()
plt.show()

Observation:

Most customers purchase small quantities of items.
The boxplot shows the spread of quantity values and possible outliers.

In [ ]:
# Countplot for Category

plt.figure(figsize=(10,6))

sns.countplot(data=clean_df, x="Category", order=clean_df["Category"].value_counts().index)

plt.title("Number of Transactions by Category")
plt.xlabel("Category")
plt.ylabel("Count")

plt.xticks(rotation=90)
plt.show()

Observation:

This bar chart shows the number of transactions in each product category.
Some categories are more popular and have higher transaction counts.

In [ ]:
# Countplot for Payment Method

plt.figure(figsize=(8,5))

sns.countplot(data=clean_df, x="Payment Method",
              order=clean_df["Payment Method"].value_counts().index)

plt.title("Payment Method Distribution")
plt.xlabel("Payment Method")
plt.ylabel("Count")

plt.show()

Observation:

This chart shows the most commonly used payment methods.
It helps identify customer payment preferences.

In [ ]:
# Pie chart for Location distribution

location_counts = clean_df["Location"].value_counts()

plt.figure(figsize=(8,8))

plt.pie(location_counts,
        labels=location_counts.index,
        autopct='%1.1f%%',
        startangle=90)

plt.title("Sales Distribution by Location")

plt.show()

Observation:

This pie chart shows the percentage of transactions by location.
Some locations contribute more sales than others.

In [ ]:
# Pie chart for Discount Applied

discount_counts = clean_df["Discount Applied"].value_counts()

plt.figure(figsize=(6,6))

plt.pie(discount_counts,
        labels=discount_counts.index,
        autopct='%1.1f%%',
        startangle=90)

plt.title("Discount Applied Distribution")

plt.show()

Observation:

This chart shows the proportion of transactions with and without discounts.
Most transactions may occur without discounts depending on customer behavior.

####Bivariate Analysis


In [ ]:
# Boxplot: Quantity vs Total Spent

plt.figure(figsize=(10,6))

sns.boxplot(data=clean_df,
            x="Quantity",
            y="Total Spent")

plt.title("Quantity vs Total Spent")
plt.xlabel("Quantity")
plt.ylabel("Total Spent")

plt.show()

Observation:

This boxplot compares the distribution of Total Spent across different quantity levels.
Customers who purchase higher quantities generally have higher total spending.
The plot also helps identify spending variation and possible outliers for each quantity group.

In [ ]:
# Boxplot for Total Spent by Category

plt.figure(figsize=(10,6))

sns.boxplot(data=clean_df,
            x="Category",
            y="Total Spent")

plt.title("Total Spent by Category")
plt.xlabel("Category")
plt.ylabel("Total Spent")

plt.xticks(rotation=90)

plt.show()

Observation:

This boxplot compares customer spending across categories.
Some categories have higher median spending and wider spending ranges.

In [ ]:
# Boxplot: Total Spent by Payment Method

plt.figure(figsize=(10,6))

sns.boxplot(data=clean_df,
            x="Payment Method",
            y="Total Spent")

plt.title("Total Spent by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Total Spent")

plt.show()

Observation:

This boxplot compares customer spending across different payment methods.
It shows the median spending, spread of values, and possible outliers for each payment type.
Some payment methods may have higher average customer spending than others.

####Multivariate Analysis


In [ ]:
# Correlation heatmap

plt.figure(figsize=(8,6))


numeric_df = clean_df.select_dtypes(include=["int64", "float64"])

sns.heatmap(numeric_df.corr(),
            annot=True,
            cmap="coolwarm",
            fmt=".2f")

plt.title("Correlation Heatmap")

plt.show()

Observation:

The heatmap shows relationships between numerical variables.
Total Spent has a strong positive correlation with Quantity and Price Per Unit.

In [ ]:
# Stacked bar chart for Category and Payment Method

cross_tab = pd.crosstab(clean_df["Category"],
                        clean_df["Payment Method"])

cross_tab.plot(kind="bar",
               stacked=True,
               figsize=(10,6))

plt.title("Category vs Payment Method")
plt.xlabel("Category")
plt.ylabel("Count")

plt.xticks(rotation=90)

plt.show()

Observation:

This stacked bar chart shows the relationship between product categories and payment methods.
It helps identify preferred payment methods for different categories.

In [ ]:
# Average spending by Location and Payment Method

grouped_data = clean_df.groupby(
    ["Location", "Payment Method"]
)["Total Spent"].mean().unstack()

grouped_data.plot(kind="bar",
                  figsize=(10,6))

plt.title("Average Total Spent by Location and Payment Method")
plt.xlabel("Location")
plt.ylabel("Average Total Spent")

plt.xticks(rotation=45)

plt.show()

Observation:

This grouped bar chart compares average spending across locations and payment methods.
It helps identify customer spending behavior in different regions.

###Feature Engineering


In [ ]:
# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

# Feature 1: Transaction Month
clean_df["Transaction Month"] = clean_df["Transaction Date"].dt.month

# Feature 2: Spending Category Target
clean_df["Spending Category"] = pd.cut(
    clean_df["Total Spent"],
    bins=[-1, 50, 150, 500],
    labels=["Low", "Medium", "High"]
)

# Check new features
clean_df[["Transaction Date", "Transaction Month", "Total Spent", "Spending Category"]].head()

In [ ]:
# Check target distribution
clean_df["Spending Category"].value_counts()

In [ ]:
# Check missing values after feature engineering
clean_df.isnull().sum()

In [ ]:
# ==========================================================
# IMPORT MACHINE LEARNING LIBRARIES
# ==========================================================

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, validation_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# ==========================================================
# PREPARE DATA FOR MODELING
# ==========================================================

model_df = clean_df.copy()

# Drop rows with missing target
model_df = model_df.dropna(subset=["Spending Category"])

# Drop columns not needed for modeling
X = model_df.drop(
    columns=[
        "Spending Category",
        "Transaction ID",
        "Customer ID",
        "Transaction Date",
        "Total Spent"
    ],
    errors="ignore"
)

y = model_df["Spending Category"]

# Convert categorical features into dummy variables
X = pd.get_dummies(X, drop_first=True)

# Encode target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Feature shape:", X.shape)
print("Target classes:", le.classes_)

In [ ]:
# ==========================================================
# TRAIN-TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# ==========================================================
# FEATURE SCALING
# ==========================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ==========================================================
# PCA DIMENSIONALITY REDUCTION
# ==========================================================

pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original number of features:", X_train_scaled.shape[1])
print("Number of PCA components:", X_train_pca.shape[1])

In [ ]:
# ==========================================================
# TRAIN 4 MACHINE LEARNING MODELS
# ==========================================================

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

results = []

for name, model in models.items():

    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

    print("="*60)
    print(name)
    print("="*60)
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# ==========================================================
# MODEL COMPARISON TABLE
# ==========================================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-Score", ascending=False)

results_df

In [ ]:
# ==========================================================
# MODEL PERFORMANCE COMPARISON CHART
# ==========================================================

plt.figure(figsize=(10,6))

sns.barplot(
    data=results_df,
    x="Model",
    y="F1-Score"
)

plt.title("Model Performance Comparison Based on F1-Score")
plt.xlabel("Machine Learning Model")
plt.ylabel("F1-Score")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# ==========================================================
# CROSS-VALIDATION
# ==========================================================

cv_results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_pca,
        y_train,
        cv=5,
        scoring="f1_weighted"
    )

    cv_results.append({
        "Model": name,
        "CV Mean F1-Score": scores.mean(),
        "CV Standard Deviation": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df

In [ ]:
# ==========================================================
# HYPERPARAMETER TUNING - RANDOM FOREST
# ==========================================================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid_search.fit(X_train_pca, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV F1-Score:")
print(grid_search.best_score_)

In [ ]:
# ==========================================================
# EVALUATE TUNED RANDOM FOREST
# ==========================================================

best_rf = grid_search.best_estimator_

y_pred_tuned = best_rf.predict(X_test_pca)

print(classification_report(y_test, y_pred_tuned, target_names=le.classes_, zero_division=0))

tuned_result = {
    "Model": "Tuned Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "Precision": precision_score(y_test, y_pred_tuned, average="weighted", zero_division=0),
    "Recall": recall_score(y_test, y_pred_tuned, average="weighted", zero_division=0),
    "F1-Score": f1_score(y_test, y_pred_tuned, average="weighted", zero_division=0)
}

tuned_result

In [ ]:
# ==========================================================
# CONFUSION MATRIX - TUNED RANDOM FOREST
# ==========================================================

cm = confusion_matrix(y_test, y_pred_tuned)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Tuned Random Forest")
plt.show()

In [ ]:
# ==========================================================
# VALIDATION CURVE FOR KNN
# ==========================================================

param_range = [1, 3, 5, 7, 9, 11, 13, 15]

train_scores, test_scores = validation_curve(
    KNeighborsClassifier(),
    X_train_pca,
    y_train,
    param_name="n_neighbors",
    param_range=param_range,
    cv=5,
    scoring="f1_weighted"
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(10,6))

plt.plot(param_range, train_mean, marker="o", label="Training Score")
plt.plot(param_range, test_mean, marker="o", label="Validation Score")

plt.title("Validation Curve for KNN")
plt.xlabel("Number of Neighbors")
plt.ylabel("F1-Score")
plt.legend()
plt.show()

In [ ]:
# ==========================================================
# FINAL MODEL COMPARISON
# ==========================================================

final_results = pd.concat([
    results_df,
    pd.DataFrame([tuned_result])
], ignore_index=True)

final_results = final_results.sort_values(by="F1-Score", ascending=False)

final_results

In [ ]:
# ==========================================================
# FINAL MODEL COMPARISON CHART
# ==========================================================

plt.figure(figsize=(12,6))

sns.barplot(
    data=final_results,
    x="Model",
    y="F1-Score"
)

plt.title("Final Model Comparison Based on F1-Score")
plt.xlabel("Model")
plt.ylabel("F1-Score")
plt.xticks(rotation=30)
plt.show()